# Spot Tolerance Cost Analysis — Scenario B

In [ ]:
import json
import os
import re
import glob
from collections import defaultdict
from itertools import groupby

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

# ── Configuration ──────────────────────────────────────────────────────────────
SCENARIO = "B"

MODEL_CONFIGS = {
    "llama3-70b": {
        "benchmark_duration_min": 60,
        "az_selection": {},
    },
}

APPROACHES = ["only_ondemand", "shuntserve", "concurrent_initialization",
              "request_migration", "no_handle"]
OVERLAP_APPROACHES = {"shuntserve", "concurrent_initialization"}

FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Display config ────────────────────────────────────────────────────────────
DISPLAY_NAMES = {
    "only_ondemand": "On-demand\nOnly",
    "no_handle": "No\nHandle",
    "request_migration": "Request\nMigration",
    "concurrent_initialization": "Concurrent\nInitialization",
    "shuntserve": "ShuntServe",
}
PLOT_ORDER = ["only_ondemand", "no_handle", "request_migration",
              "concurrent_initialization", "shuntserve"]
COLORS = {
    "only_ondemand": "#808080",
    "no_handle": "#d62728",
    "request_migration": "#2ca02c",
    "concurrent_initialization": "#ff7f0e",
    "shuntserve": "#1f77b4",
}
HATCHES = {
    "only_ondemand": "",
    "no_handle": "/",
    "request_migration": "\\",
    "concurrent_initialization": "x",
    "shuntserve": ".",
}

print(f"Scenario: {SCENARIO}")
for mk, cfg in MODEL_CONFIGS.items():
    print(f"  {mk}: duration={cfg['benchmark_duration_min']}min")

In [ ]:
# ── Helper Functions ───────────────────────────────────────────────────────────

def node_to_group(name: str) -> str:
    """spot_g6e_xlarge_node_ip_1 → spot_g6e_xlarge"""
    return re.sub(r"_node_ip_\d+$", "", name)

def group_to_instance_type(group: str) -> str:
    """spot_g6e_xlarge → g6e.xlarge"""
    bare = re.sub(r"^(spot|on_demand)_", "", group)
    return re.sub(r"_(\d*xlarge)$", r".\1", bare)

def is_spot_group(group: str) -> bool:
    return group.startswith("spot_")


# ── load_data ─────────────────────────────────────────────────────────────────

def load_data(model_key: str, scenario: str) -> dict:
    """Load all data files and compute initial state."""
    mk_underscore = model_key.replace("-", "_")

    with open(f"results/prices_scenario_{scenario}.json") as f:
        prices_data = json.load(f)
    with open(f"spot_trace_events_scenario_{scenario}.json") as f:
        events_raw = json.load(f)["events"]
    with open(f"nodes_scenario_{scenario}.json") as f:
        nodes_data = json.load(f)
    with open(f"{model_key}/pipelines_{mk_underscore}_scenario_{scenario}.json") as f:
        pipeline_data = json.load(f)

    # Parse node groups
    all_node_groups = defaultdict(int)
    for name in nodes_data:
        all_node_groups[node_to_group(name)] += 1

    # Parse initial active nodes from pipeline
    initial_active = defaultdict(int)
    for pipeline in pipeline_data["pipelines"]:
        for node_name, _layers in pipeline["node_layer_mapping"]:
            initial_active[node_to_group(node_name)] += 1

    # Per instance type: initial (spot_active, od_active)
    instance_types = set(group_to_instance_type(g) for g in all_node_groups)
    initial_state = {}
    for itype in sorted(instance_types):
        spot_total = od_total = spot_active = od_active = 0
        for g, cnt in all_node_groups.items():
            if group_to_instance_type(g) == itype:
                if is_spot_group(g):
                    spot_total += cnt
                    spot_active += initial_active.get(g, 0)
                else:
                    od_total += cnt
                    od_active += initial_active.get(g, 0)
        initial_state[itype] = {
            "spot_active": spot_active, "od_active": od_active,
            "spot_total": spot_total, "od_total": od_total,
        }

    print(f"[{model_key}] Node groups: {dict(sorted(all_node_groups.items()))}")
    print(f"[{model_key}] Initial state: {initial_state}")

    return {
        "prices_data": prices_data,
        "events_raw": events_raw,
        "nodes_data": nodes_data,
        "pipeline_data": pipeline_data,
        "all_node_groups": dict(all_node_groups),
        "instance_types": instance_types,
        "initial_state": initial_state,
        "initial_active": dict(initial_active),
    }

In [ ]:
# ── parse_metrics ──────────────────────────────────────────────────────────────

def _parse_switching_times(log_path: str) -> list[float]:
    times = []
    with open(log_path) as f:
        for line in f:
            m = re.search(r"(?:switch completed|recreated) in (\d+\.?\d*)s", line)
            if m:
                times.append(float(m.group(1)))
    return times

def _find_trace_csv(approach: str, trace_dir: str, prefix: str,
                    mk_underscore: str, scenario: str) -> str | None:
    patterns = [
        f"{trace_dir}/spottolerance_{prefix}_{approach}_{mk_underscore}_scenario_{scenario}_*.csv",
        f"{trace_dir}/spottolerance_{prefix}_{approach}_scenario_{scenario}_*.csv",
    ]
    for pat in patterns:
        matches = sorted(glob.glob(pat))
        if matches:
            return matches[-1]
    return None

def _calc_throughput(csv_path: str, duration_min: int) -> float:
    df = pd.read_csv(csv_path)
    t0 = df["ArrivalTime"].min()
    cutoff = t0 + duration_min * 60
    return len(df[df["CompletionTime"] <= cutoff]) / (duration_min * 60)

def _calc_mean_latency(csv_path: str, duration_min: int) -> float:
    df = pd.read_csv(csv_path)
    t0 = df["ArrivalTime"].min()
    cutoff = t0 + duration_min * 60
    return df[df["CompletionTime"] <= cutoff]["Latency"].mean()


def parse_metrics(model_key: str, scenario: str, benchmark_duration_min: int,
                  approaches: list[str]) -> dict:
    """Parse switching times, offline throughputs, online metrics."""
    mk_underscore = model_key.replace("-", "_")
    log_dir = f"{model_key}/offline/scenario_{scenario}/logs"
    trace_dir = f"results/{model_key}/offline/scenario_{scenario}/Trace"
    online_trace_dir = f"results/{model_key}/online/scenario_{scenario}/Trace"

    switching_times = {}
    throughputs = {}
    online_throughputs = {}
    online_mean_latencies = {}

    for approach in approaches:
        # Switching times from logs
        log_path = os.path.join(log_dir, f"{approach}.log")
        if os.path.exists(log_path):
            st = _parse_switching_times(log_path)
            if st:
                switching_times[approach] = st

        # Offline throughput from CSV
        csv_path = _find_trace_csv(approach, trace_dir, "offline", mk_underscore, scenario)
        if csv_path:
            throughputs[approach] = _calc_throughput(csv_path, benchmark_duration_min)

        # Online metrics from CSV
        csv_path = _find_trace_csv(approach, online_trace_dir, "online", mk_underscore, scenario)
        if csv_path:
            online_throughputs[approach] = _calc_throughput(csv_path, benchmark_duration_min)
            online_mean_latencies[approach] = _calc_mean_latency(csv_path, benchmark_duration_min)

    print(f"[{model_key}] Switching times: {switching_times}")
    print(f"[{model_key}] Offline throughputs: { {a: f'{v:.4f}' for a,v in throughputs.items()} }")
    if online_mean_latencies:
        print(f"[{model_key}] Online mean latencies: { {a: f'{v:.2f}' for a,v in online_mean_latencies.items()} }")

    return {
        "switching_times": switching_times,
        "throughputs": throughputs,
        "online_throughputs": online_throughputs,
        "online_mean_latencies": online_mean_latencies,
    }

In [ ]:
# ── compute_active_periods ────────────────────────────────────────────────────

def compute_active_periods(events_raw: list, initial_state: dict,
                           instance_types: set, benchmark_duration_min: int) -> dict:
    """Compute timeline segments and compound outgoing nodes."""

    events = [e for e in events_raw if e["time_min"] < benchmark_duration_min]
    events_sorted = sorted(events, key=lambda e: e["time_min"])
    compound_events = []
    for t, grp in groupby(events_sorted, key=lambda e: e["time_min"]):
        compound_events.append({"time_min": t, "sub_events": list(grp)})

    def _count_spot_nodes(sub_event):
        counts = defaultdict(int)
        for inst in sub_event["instances"]:
            group = node_to_group(inst)
            if is_spot_group(group):
                counts[group_to_instance_type(group)] += 1
        return dict(counts)

    # Build timeline segments
    state = {i: {"spot": s["spot_active"], "od": s["od_active"]}
             for i, s in initial_state.items()}
    segments = {i: [] for i in instance_types}
    last_time = 0

    for ce in compound_events:
        t = ce["time_min"]
        if t > last_time:
            for i in instance_types:
                segments[i].append((last_time, t, state[i]["spot"], state[i]["od"]))
        for se in ce["sub_events"]:
            for itype, n in _count_spot_nodes(se).items():
                if se["type"] == "interruption":
                    state[itype]["spot"] -= n
                    state[itype]["od"] += n
                else:
                    state[itype]["spot"] += n
                    state[itype]["od"] -= n
        last_time = t

    for i in instance_types:
        segments[i].append((last_time, benchmark_duration_min, state[i]["spot"], state[i]["od"]))

    # Compute outgoing nodes per compound event (for overlap cost)
    overlap_state = {i: {"spot": s["spot_active"], "od": s["od_active"]}
                     for i, s in initial_state.items()}
    compound_outgoing = []
    for ce in compound_events:
        outgoing = {}
        for se in ce["sub_events"]:
            for itype, n in _count_spot_nodes(se).items():
                if se["type"] == "interruption":
                    outgoing[itype] = {"count": n, "is_spot": True}
                    overlap_state[itype]["spot"] -= n
                    overlap_state[itype]["od"] += n
                else:
                    outgoing[itype] = {"count": n, "is_spot": False}
                    overlap_state[itype]["spot"] += n
                    overlap_state[itype]["od"] -= n
        compound_outgoing.append(outgoing)

    print(f"  Compound events: {len(compound_events)}")
    return {
        "segments": segments,
        "compound_events": compound_events,
        "compound_outgoing": compound_outgoing,
    }

In [ ]:
# ── Price Lookup ───────────────────────────────────────────────────────────────

def _get_spot_prices_by_az(prices_data: dict, instance_type: str) -> dict[str, float]:
    az_prices = defaultdict(list)
    for entry in prices_data["spot"]:
        if entry["Instance"] == instance_type:
            az_prices[entry["AZ"]].append(float(entry["Price"]))
    return {az: sum(ps) / len(ps) for az, ps in az_prices.items()}

def _get_ondemand_price(prices_data: dict, instance_type: str) -> float:
    for entry in prices_data["ondemand"]:
        if entry["Instance"] == instance_type:
            return float(entry["PricePerHour_USD"])
    raise ValueError(f"On-demand price not found for {instance_type}")

def _get_spot_price(prices_data: dict, instance_type: str,
                    az_selection: dict, mode: str = "avg") -> float:
    az_prices = _get_spot_prices_by_az(prices_data, instance_type)
    if not az_prices:
        raise ValueError(f"No spot price data for {instance_type}")
    if instance_type in az_selection and az_selection[instance_type]:
        az = az_selection[instance_type]
        if az in az_prices:
            return az_prices[az]
        raise ValueError(f"AZ {az} not found for {instance_type}")
    all_prices = list(az_prices.values())
    if mode == "avg":
        return sum(all_prices) / len(all_prices)
    elif mode == "min":
        return min(all_prices)
    elif mode == "max":
        return max(all_prices)
    raise ValueError(f"Unknown mode: {mode}")

In [ ]:
# ── Cost Calculation ───────────────────────────────────────────────────────────

def calculate_all_costs(data: dict, periods: dict, metrics: dict,
                        az_selection: dict, benchmark_duration_min: int,
                        approaches: list[str], mode: str = "avg") -> dict[str, float]:
    """Calculate cost for each approach under a given spot price mode."""
    prices_data = data["prices_data"]
    initial_state = data["initial_state"]
    instance_types = data["instance_types"]
    segments = periods["segments"]
    compound_outgoing = periods["compound_outgoing"]
    switching_times = metrics["switching_times"]

    # 1. ondemand_only: all initial pipeline nodes at on-demand price
    pipeline_node_counts = defaultdict(int)
    for itype in instance_types:
        s = initial_state[itype]
        pipeline_node_counts[itype] = s["spot_active"] + s["od_active"]

    ondemand_cost = 0.0
    for itype, count in pipeline_node_counts.items():
        if count > 0:
            ondemand_cost += count * _get_ondemand_price(prices_data, itype) * benchmark_duration_min / 60

    # 2. Base mixed cost
    base_cost = 0.0
    for itype in instance_types:
        spot_price = _get_spot_price(prices_data, itype, az_selection, mode) if initial_state[itype]["spot_total"] > 0 else 0
        od_price = _get_ondemand_price(prices_data, itype) if initial_state[itype]["od_total"] > 0 else 0
        for t_start, t_end, spot_n, od_n in segments[itype]:
            duration_hr = (t_end - t_start) / 60
            base_cost += spot_n * spot_price * duration_hr
            base_cost += od_n * od_price * duration_hr

    # 3. Overlap cost per approach
    results = {"only_ondemand": ondemand_cost}
    for approach in approaches:
        if approach == "only_ondemand":
            continue
        total_cost = base_cost
        if approach in OVERLAP_APPROACHES and approach in switching_times:
            for i, outgoing in enumerate(compound_outgoing):
                if i < len(switching_times[approach]):
                    switch_sec = switching_times[approach][i]
                    for itype, info in outgoing.items():
                        if info["is_spot"]:
                            price = _get_spot_price(prices_data, itype, az_selection, mode)
                        else:
                            price = _get_ondemand_price(prices_data, itype)
                        total_cost += info["count"] * price * switch_sec / 3600
        results[approach] = total_cost

    return results


def compute_costs(data: dict, periods: dict, metrics: dict,
                  az_selection: dict, benchmark_duration_min: int,
                  approaches: list[str]) -> tuple[dict, list[str]]:
    """Compute costs for all applicable spot price modes."""
    instance_types = data["instance_types"]
    initial_state = data["initial_state"]

    all_az_specified = all(
        itype in az_selection and az_selection[itype]
        for itype in instance_types
        if initial_state[itype]["spot_total"] > 0
    )
    modes = ["avg"] if all_az_specified else ["avg", "min", "max"]

    cost_results = {}
    for mode in modes:
        cost_results[mode] = calculate_all_costs(
            data, periods, metrics, az_selection, benchmark_duration_min, approaches, mode
        )

    return cost_results, modes

In [ ]:
# ── Plot & Summary Functions ───────────────────────────────────────────────────

def print_summary(cost_results, modes, metrics, model_key, benchmark_duration_min):
    """Print summary table."""
    throughputs = metrics["throughputs"]
    online_mean_latencies = metrics["online_mean_latencies"]

    rows = []
    for mode in modes:
        for approach in APPROACHES:
            cost = cost_results[mode].get(approach)
            tput = throughputs.get(approach)
            lat = online_mean_latencies.get(approach)
            rows.append({
                "Approach": approach,
                "Spot Price Mode": mode,
                "Cost ($)": f"{cost:.4f}" if cost is not None else "N/A",
                "Offline Throughput (req/s)": f"{tput:.2f}" if tput is not None else "N/A",
                "Online Mean Latency (s)": f"{lat:.2f}" if lat is not None else "N/A",
            })
    display(pd.DataFrame(rows))


def plot_cost_comparison(cost_results, modes, model_key):
    """Bar chart of total cost per approach."""
    mode = modes[0]
    costs_dict = cost_results[mode]
    values = [costs_dict[a] for a in PLOT_ORDER]

    fontsize = 35
    fig, ax = plt.subplots(figsize=(8, 6))
    for i, a in enumerate(PLOT_ORDER):
        ax.bar(i, costs_dict[a], width=0.8, color=COLORS[a], alpha=0.7,
               edgecolor="black", linewidth=1.5, hatch=HATCHES[a])
    for i, v in enumerate(values):
        ax.text(i, v + max(values) * 0.02, f"{v:.2f}",
                ha="center", va="bottom", fontsize=fontsize, rotation=90)

    ax.set_ylabel("Cost ($)", fontsize=fontsize)
    ax.set_xticks(range(len(PLOT_ORDER)))
    ax.set_xticklabels([])
    ax.set_xlabel(" ", fontsize=fontsize)
    ax.set_ylim(0, max(values) * 1.45)
    ax.tick_params(axis="y", labelsize=fontsize)
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))

    mk_suffix = model_key.replace("-", "_")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/cost_comparison_scenario_{SCENARIO}_{mk_suffix}.pdf",
                dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIGURES_DIR}/cost_comparison_scenario_{SCENARIO}_{mk_suffix}.pdf")


def plot_cost_efficiency(cost_results, modes, metrics, model_key):
    """Offline+Online 2-group normalized efficiency bar chart."""
    mode = modes[0]
    costs_dict = cost_results[mode]
    throughputs = metrics["throughputs"]
    online_mean_latencies = metrics["online_mean_latencies"]

    # Offline: Throughput / Cost
    offline_eff = {}
    for a in PLOT_ORDER:
        if a in throughputs and a in costs_dict and costs_dict[a] > 0:
            offline_eff[a] = throughputs[a] / costs_dict[a]
    offline_baseline = offline_eff.get("only_ondemand", 1)
    offline_norm = {a: offline_eff[a] / offline_baseline for a in PLOT_ORDER if a in offline_eff}

    # Online: 1 / (Latency × Cost)
    online_norm = {}
    if online_mean_latencies:
        online_eff = {}
        for a in PLOT_ORDER:
            if a in online_mean_latencies and a in costs_dict and costs_dict[a] > 0:
                online_eff[a] = 1.0 / (online_mean_latencies[a] * costs_dict[a])
        online_baseline = online_eff.get("only_ondemand", 1)
        online_norm = {a: online_eff[a] / online_baseline for a in PLOT_ORDER if a in online_eff}

    has_online = bool(online_norm)

    print("Offline Throughput/Cost (normalized):")
    for a in PLOT_ORDER:
        if a in offline_norm:
            print(f"  {a:35s}  {offline_norm[a]:.3f}x")
    if has_online:
        print("\nOnline 1/(Latency×Cost) (normalized):")
        for a in PLOT_ORDER:
            if a in online_norm:
                print(f"  {a:35s}  {online_norm[a]:.3f}x")

    # Plot
    fontsize = 35
    fig, ax = plt.subplots(figsize=(8, 6))
    n_approaches = len(PLOT_ORDER)
    bar_width = 0.25
    group_gap = 0.3

    g1_positions = [i * bar_width for i in range(n_approaches)]
    g1_center = (n_approaches - 1) * bar_width / 2

    for i, a in enumerate(PLOT_ORDER):
        v = offline_norm.get(a, 0)
        ax.bar(g1_positions[i], v, bar_width, color=COLORS[a], alpha=0.7,
               edgecolor="black", linewidth=1.5, hatch=HATCHES[a])
        label = "1.00x" if a == "only_ondemand" else f"{v:.2f}x"
        ax.text(g1_positions[i], v + 0.05, label,
                ha="center", va="bottom", fontsize=fontsize - 4, rotation=90)

    if has_online:
        g2_start = n_approaches * bar_width + group_gap
        g2_positions = [g2_start + i * bar_width for i in range(n_approaches)]
        g2_center = g2_start + (n_approaches - 1) * bar_width / 2
        for i, a in enumerate(PLOT_ORDER):
            v = online_norm.get(a, 0)
            ax.bar(g2_positions[i], v, bar_width, color=COLORS[a], alpha=0.7,
                   edgecolor="black", linewidth=1.5, hatch=HATCHES[a])
            label = "1.00x" if a == "only_ondemand" else f"{v:.2f}x"
            ax.text(g2_positions[i], v + 0.05, label,
                    ha="center", va="bottom", fontsize=fontsize - 4, rotation=90)
        ax.set_xticks([g1_center, g2_center])
        ax.set_xticklabels(["Offline", "Online"], fontsize=fontsize)
    else:
        ax.set_xticks([g1_center])
        ax.set_xticklabels(["Offline"], fontsize=fontsize)

    all_norm = list(offline_norm.values()) + list(online_norm.values())
    ax.set_ylabel("Normalized Efficiency", fontsize=fontsize)
    ax.set_ylim(0, max(all_norm) * 1.4)
    ax.tick_params(axis="y", labelsize=fontsize)
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")

    mk_suffix = model_key.replace("-", "_")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/cost_efficiency_scenario_{SCENARIO}_{mk_suffix}.pdf",
                dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {FIGURES_DIR}/cost_efficiency_scenario_{SCENARIO}_{mk_suffix}.pdf")


def plot_legend(model_key):
    """Separate legend file."""
    fig, ax = plt.subplots(figsize=(10, 0.5))
    ax.axis("off")
    legend_elements = [
        mpatches.Patch(facecolor=COLORS[a], edgecolor="black", alpha=0.7,
                       hatch=HATCHES[a], label=DISPLAY_NAMES[a].replace("\n", " "))
        for a in PLOT_ORDER
    ]
    ax.legend(handles=legend_elements, fontsize=20, loc="center", ncol=5,
              columnspacing=0.8, handlelength=1.5, edgecolor="black")

    mk_suffix = model_key.replace("-", "_")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/cost_legend_scenario_{SCENARIO}_{mk_suffix}.pdf",
                dpi=300, bbox_inches="tight", transparent=True)
    plt.show()
    print(f"Saved: {FIGURES_DIR}/cost_legend_scenario_{SCENARIO}_{mk_suffix}.pdf")

In [ ]:
# ── run_analysis ───────────────────────────────────────────────────────────────

def run_analysis(model_key: str):
    """Run full cost analysis pipeline for a model."""
    cfg = MODEL_CONFIGS[model_key]
    bdm = cfg["benchmark_duration_min"]
    az_sel = cfg["az_selection"]

    print(f"\n{'='*70}")
    print(f"  {model_key} — Scenario {SCENARIO} — {bdm} min")
    print(f"{'='*70}\n")

    data = load_data(model_key, SCENARIO)
    metrics = parse_metrics(model_key, SCENARIO, bdm, APPROACHES)
    periods = compute_active_periods(data["events_raw"], data["initial_state"],
                                     data["instance_types"], bdm)
    cost_results, modes = compute_costs(data, periods, metrics, az_sel, bdm, APPROACHES)

    # Print costs
    for mode in modes:
        print(f"\nSpot price mode: {mode.upper()}")
        for approach, cost in cost_results[mode].items():
            print(f"  {approach:35s}  ${cost:.4f}")

    # Summary table
    print_summary(cost_results, modes, metrics, model_key, bdm)

    # Plots
    plot_cost_comparison(cost_results, modes, model_key)
    plot_cost_efficiency(cost_results, modes, metrics, model_key)
    plot_legend(model_key)

In [ ]:
run_analysis("llama3-70b")